# Procesamiento de Video con YOLO

Este notebook está diseñado únicamente para procesar un video utilizando los 3 modelos integrados:
1. **Detección OBB** (modelo personalizado de tenis de mesa entrenado con YOLO26 OBB).
2. **Segmentación de Objetos** (yolo26n-seg.pt de COCO, excluyendo personas).
3. **Pose Estimation** (yolo26n-pose.pt para el esqueleto de los jugadores).

El procesamiento está limitado para extraer y analizar únicamente **10 segundos** del video original.

### 1. Instalación de Librerías
Instalamos los paquetes necesarios para procesar video e inferencia de modelos.

In [1]:
%pip install ultralytics opencv-python numpy pyyaml matplotlib

Note: you may need to restart the kernel to use updated packages.


### 2. Parámetros de Confianza Ajustables
Modifica los siguientes valores para calibrar la sensibilidad de cada modelo según necesites. Si bajas el valor, detectará más objetos pero con mayor probabilidad de falsos positivos.

In [1]:
# Umbrales de confianza mínimos para cada modelo (rango de 0.0 a 1.0)
CONF_DETECTION_CLASS = {
    'TT TABLE': 0.80,   # Mesa
    'TT NET': 0.50,     # Red
    'TT RACKET': 0.19   # Paleta
}
MIN_CONF_DETECTION = min(CONF_DETECTION_CLASS.values())
CONF_SEG = 0.30  # Para el modelo de segmentación de objetos COCO
CONF_POSE = 0.25  # Para el modelo de pose (esqueleto de personas)

print("Parámetros de confianza configurados:")
print(f"Detección OBB (Mínima): {MIN_CONF_DETECTION}")
print(f"Segmentación: {CONF_SEG}")
print(f"Pose: {CONF_POSE}")

Parámetros de confianza configurados:
Detección OBB (Mínima): 0.19
Segmentación: 0.3
Pose: 0.25


### 3. Cargar Modelos
Buscamos el mejor modelo entrenado (`best.pt`) y cargamos los modelos preentrenados de Segmentación y Pose de **YOLO26**.

In [2]:
import os
import glob
from collections import defaultdict
import numpy as np
import cv2
from ultralytics import YOLO

# Buscar dinámicamente los mejores pesos entrenados
best_weights = glob.glob('**/weights/best.pt', recursive=True)
if best_weights:
    BEST_WEIGHTS_PATH = best_weights[0]
    print(f"Modelo personalizado cargado desde: {BEST_WEIGHTS_PATH}")
else:
    BEST_WEIGHTS_PATH = 'yolo26n-obb.pt'
    print(f"No se encontró best.pt, usando modelo base: {BEST_WEIGHTS_PATH}")

model_detection    = YOLO(BEST_WEIGHTS_PATH)
model_segmentation = YOLO('yolo26n-seg.pt')
model_pose         = YOLO('yolo26n-pose.pt')

print("Modelos cargados correctamente")

Modelo personalizado cargado desde: runs/obb/runs/tenis_mesa_obb_run/weights/best.pt
Modelos cargados correctamente


### 4. Configuración del Video de Entrada/Salida
Busca los videos disponibles en el directorio `Videos/` y configura el video resultante de 10 segundos.

In [3]:
# Buscar videos en la carpeta Videos/
videos = glob.glob('Videos/*.mp4')

if videos:
    VIDEO_INPUT = videos[0]
    print(f"Video encontrado para procesar: {VIDEO_INPUT}")
else:
    VIDEO_INPUT = "video1.mp4"
    print(f"No se encontró la carpeta Videos/, usando fallback: {VIDEO_INPUT}")

VIDEO_OUTPUT = 'video_procesado_10s.mp4'
print(f"El video procesado se guardará en: {VIDEO_OUTPUT}")

Video encontrado para procesar: Videos/TT_Olimpic.mp4
El video procesado se guardará en: video_procesado_10s.mp4


### 5. Definición de Funciones de Dibujo (Visualización)
Configuramos colores personalizados y lógica para dibujar bounding boxes orientadas (OBB), segmentación COCO (sin personas), pose estimation e información en pantalla.

In [4]:
# Configuración de Clases y Panel HUD
import time
import numpy as np
import cv2
from collections import defaultdict

# ID de la clase de persona en COCO (para excluirla de la segmentación)
COCO_PERSON_CLASS_ID = 0

# Lista de clases para la segmentación (todas excepto personas)
CLASES_SEG_INCLUIDAS = [i for i in range(80) if i != COCO_PERSON_CLASS_ID]

# Colores para nuestras clases personalizadas
CUSTOM_CLASS_COLORS = {
    'tt net':    (252, 191, 73),   # Naranja/Amarillo
    'tt racket': (0, 180, 216),    # Celeste
    'tt table':  (255, 0, 110),    # Rosa
}

# Nombres a mostrar en el panel
DISPLAY_NAMES = {
    'tt net':    'TT Net',
    'tt racket': 'TT Racket',
    'tt table':  'TT Table',
}
ALL_CUSTOM_CLASSES = ['TT Racket', 'TT Net', 'TT Table']


def extraer_conteo_detecciones(det_result, conf_thresh_dict):
    # Cuenta las detecciones filtrando por confianza
    counts = defaultdict(int)

    is_obb = hasattr(det_result, 'obb') and det_result.obb is not None and len(det_result.obb) > 0
    is_boxes = hasattr(det_result, 'boxes') and det_result.boxes is not None and len(det_result.boxes) > 0

    if not is_obb and not is_boxes:
        return dict(counts)

    detections = det_result.obb if is_obb else det_result.boxes

    for box in detections:
        cls_id     = int(box.cls[0])
        confidence = float(box.conf[0])
        cls_name   = det_result.names[cls_id]

        # Umbral especifico por clase
        normalized_name = cls_name.upper().strip()
        thresh = conf_thresh_dict.get(normalized_name, 0.25)
        if confidence < thresh:
            continue

        normalized = cls_name.lower().strip()
        display    = DISPLAY_NAMES.get(normalized, cls_name)
        counts[display] += 1

    return dict(counts)


def draw_info_panel(frame, counts, fps, frame_idx, n_persons):
    # Dibuja un HUD semitransparente con los conteos y FPS
    h, w = frame.shape[:2]

    # Tamaño y padding del panel
    line_h     = 30
    pad_x      = 16
    pad_y      = 14
    header_h   = 36
    sep_h      = 12
    dot_radius = 5

    n_lines = len(ALL_CUSTOM_CLASSES) + 1
    panel_h = pad_y + header_h + sep_h + (n_lines * line_h) + sep_h + line_h + pad_y
    panel_w = 260

    x0, y0 = 12, 12

    # Fondo semitransparente
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + panel_w, y0 + panel_h), (18, 18, 24), -1)
    cv2.rectangle(overlay, (x0, y0), (x0 + panel_w, y0 + panel_h), (55, 55, 65), 1)
    cv2.addWeighted(overlay, 0.78, frame, 0.22, 0, frame)

    # Título del panel
    header_y = y0 + pad_y + 20
    cv2.putText(
        frame, "DETECCIONES EN TIEMPO REAL",
        (x0 + pad_x, header_y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.48, (200, 200, 210), 1, cv2.LINE_AA
    )

    # Separador
    sep_y = header_y + sep_h
    cv2.line(
        frame,
        (x0 + pad_x, sep_y), (x0 + panel_w - pad_x, sep_y),
        (70, 70, 80), 1
    )

    # Mostrar clases de tenis de mesa
    y = sep_y + 24
    for cls_name in ALL_CUSTOM_CLASSES:
        count = counts.get(cls_name, 0)
        color = CUSTOM_CLASS_COLORS.get(cls_name.lower().strip(), (200, 200, 200))

        cv2.circle(frame, (x0 + pad_x + dot_radius, y - 4), dot_radius, color, -1, cv2.LINE_AA)
        text = f"{cls_name}: {count}"
        cv2.putText(
            frame, text,
            (x0 + pad_x + dot_radius * 2 + 10, y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.50, (235, 235, 240), 1, cv2.LINE_AA
        )
        y += line_h

    # Conteo de personas
    cv2.circle(frame, (x0 + pad_x + dot_radius, y - 4), dot_radius, (0, 255, 255), -1, cv2.LINE_AA)
    cv2.putText(
        frame, f"Personas: {n_persons}",
        (x0 + pad_x + dot_radius * 2 + 10, y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 255, 255), 1, cv2.LINE_AA
    )
    y += line_h

    # FPS y número de frame
    sep_y2 = y + 2
    cv2.line(
        frame,
        (x0 + pad_x, sep_y2), (x0 + panel_w - pad_x, sep_y2),
        (50, 50, 60), 1
    )
    y = sep_y2 + 20
    cv2.putText(
        frame, f"FPS: {fps:.1f}  |  Frame: {frame_idx}",
        (x0 + pad_x, y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.40, (120, 120, 135), 1, cv2.LINE_AA
    )

    return frame


print("Funciones de visualización y HUD cargadas correctamente")


Funciones de visualización y HUD cargadas correctamente


### 6. Pipeline de Procesamiento (Límite 10 segundos)
Corremos el pipeline que lee el video, ejecuta inferencia con los 3 modelos, ensambla el panel de visualización y guarda los primeros 10 segundos en un nuevo archivo.

In [5]:
# Celda 6: Pipeline de Procesamiento de Video

cap = cv2.VideoCapture(VIDEO_INPUT)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el video: {VIDEO_INPUT}")

fps_in  = cap.get(cv2.CAP_PROP_FPS) or 30
width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Duración máxima a procesar
SEGUNDOS_A_PROCESAR = 120
limit_frames = min(int(SEGUNDOS_A_PROCESAR * fps_in), total)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps_in, (width, height))

frame_idx = 0
print(f"Procesando hasta {SEGUNDOS_A_PROCESAR}s de video ({limit_frames} frames)...\n")

try:
    while cap.isOpened() and frame_idx < limit_frames:
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.time()

        # Paso 1: Segmentación (excluyendo personas)
        result_seg = model_segmentation.predict(
            frame,
            conf=CONF_SEG,
            iou=0.45,
            imgsz=640,
            classes=CLASES_SEG_INCLUIDAS,
            verbose=False
        )[0]

        # Graficar segmentación sin bboxes para no sobrecargar
        frame_seg = result_seg.plot(
            boxes=False,
            labels=True,
            conf=True
        )

        # Paso 2: Pose estimation (solo para los jugadores)
        result_pose = model_pose.predict(
            frame_seg,
            conf=CONF_POSE,
            iou=0.45,
            imgsz=640,
            verbose=False
        )[0]

        # Contamos personas
        n_persons = 0
        if result_pose.boxes is not None:
            n_persons = sum(1 for c in result_pose.boxes.conf if float(c) >= CONF_POSE)

        # Dibujar keypoints y esqueleto
        frame_pose = result_pose.plot(
            boxes=False,
            labels=False,
            conf=False,
            kpt_radius=5
        )

        # Paso 3: Detección de elementos con modelo personalizado (OBB)
        result_det = model_detection.predict(
            frame_pose,
            conf=MIN_CONF_DETECTION,
            iou=0.45,
            imgsz=640,
            verbose=False
        )[0]

        custom_counts = extraer_conteo_detecciones(result_det, CONF_DETECTION_CLASS)

        # Graficar bboxes del modelo OBB
        frame_det = result_det.plot(
            boxes=True,
            labels=True,
            conf=True,
            line_width=2
        )

        # Paso 4: Agregar HUD informativo
        elapsed_fps = 1.0 / (time.time() - t0 + 1e-9)
        frame_final = draw_info_panel(
            frame_det,
            custom_counts,
            elapsed_fps,
            frame_idx,
            n_persons
        )

        # Guardar frame
        out.write(frame_final)
        frame_idx += 1

        # Reporte de progreso cada 30 frames
        if frame_idx % 30 == 0:
            pct = frame_idx / limit_frames * 100
            print(f"  Frame {frame_idx:04d}/{limit_frames} ({pct:.0f}%) — {elapsed_fps:.1f} FPS")

finally:
    cap.release()
    out.release()

print(f"\nCompletado. Video procesado guardado en: {VIDEO_OUTPUT}")
print(f"Total de frames procesados: {frame_idx}")


Procesando hasta 120s de video (6000 frames)...

  Frame 0030/6000 (0%) — 12.4 FPS
  Frame 0060/6000 (1%) — 15.9 FPS
  Frame 0090/6000 (2%) — 15.5 FPS
  Frame 0120/6000 (2%) — 11.1 FPS
  Frame 0150/6000 (2%) — 15.6 FPS
  Frame 0180/6000 (3%) — 15.6 FPS
  Frame 0210/6000 (4%) — 12.8 FPS
  Frame 0240/6000 (4%) — 11.6 FPS
  Frame 0270/6000 (4%) — 11.7 FPS
  Frame 0300/6000 (5%) — 13.3 FPS
  Frame 0330/6000 (6%) — 12.8 FPS
  Frame 0360/6000 (6%) — 11.8 FPS
  Frame 0390/6000 (6%) — 12.2 FPS
  Frame 0420/6000 (7%) — 11.9 FPS
  Frame 0450/6000 (8%) — 12.2 FPS
  Frame 0480/6000 (8%) — 10.4 FPS
  Frame 0510/6000 (8%) — 12.1 FPS
  Frame 0540/6000 (9%) — 11.5 FPS
  Frame 0570/6000 (10%) — 10.9 FPS
  Frame 0600/6000 (10%) — 8.5 FPS
  Frame 0630/6000 (10%) — 12.1 FPS
  Frame 0660/6000 (11%) — 10.8 FPS
  Frame 0690/6000 (12%) — 12.8 FPS
  Frame 0720/6000 (12%) — 18.0 FPS
  Frame 0750/6000 (12%) — 17.9 FPS
  Frame 0780/6000 (13%) — 18.4 FPS
  Frame 0810/6000 (14%) — 16.7 FPS
  Frame 0840/6000 (14%) —